# 10. Convolución en 1D y 2D

**Objetivos de aprendizaje.** Al terminar podrás:

1. Calcular una convolución en 1D de dos maneras: invirtiendo y deslizando, o sumando copias desplazadas.
2. Predecir qué le hace un kernel a una señal: suavizarla, derivarla o retrasarla.
3. Aplicar una convolución larga a un audio (reverberación) usando `np.convolve`.
4. Extender la convolución a dos dimensiones y calcularla en una imagen pequeña paso a paso.
5. Usar los kernels más comunes en imágenes: promedio, gaussiano, Sobel, laplaciano y realce.
6. Relacionar la convolución con la multiplicación en frecuencia, también en 2D.

**Requisito:** notebooks 05, 06 y 09. En el 06 vimos de dónde sale la convolución; aquí la practicamos a fondo.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. Repaso

$$y[n] = x[n] * h[n] = \sum_{k} x[k]\,h[n-k]$$

Hay dos maneras equivalentes de pensar esta suma:

1. **Suma de copias.** Cada muestra $x[k]$ produce una copia de $h$ desplazada a la posición $k$ y multiplicada por
   $x[k]$. La salida es la suma de todas esas copias. Es la idea del notebook 06.
2. **Invertir y deslizar.** Para calcular una sola muestra $y[n]$, se toma $h[k]$, se invierte ($h[-k]$), se desplaza
   $n$ posiciones ($h[n-k]$), se multiplica muestra a muestra por $x[k]$ y se suman los productos.

La segunda forma es la que usamos cuando calculamos a mano, así que vamos a verla con calma.

## 2. Invertir y deslizar, paso a paso

Usaremos $x[n] = \{\underset{\uparrow}{1}, 2, 3, 1\}$ y $h[n] = \{\underset{\uparrow}{1}, 0.5, 0.25\}$.
Para cada $n$ dibujamos tres cosas: $x[k]$, la versión invertida y desplazada $h[n-k]$, y su producto.
Las operaciones de invertir y desplazar son las mismas del notebook 05.

In [ ]:
n_x = np.arange(0, 4)
x = np.array([1.0, 2.0, 3.0, 1.0])
n_h = np.arange(0, 3)
h = np.array([1.0, 0.5, 0.25])

def mostrar_paso(n, ejes):
    '''Dibuja x[k], h[n-k] y su producto para un valor de n.'''
    # h[-k]: invertir
    k_inv, h_inv = pds.invertir_tiempo(n_h, h)
    # h[n-k]: desplazar la versión invertida n lugares
    k_des, h_des = pds.desplazar(k_inv, h_inv, n)
    # producto x[k]·h[n-k] (alineando los índices) y su suma
    k_prod, producto = pds.multiplicar_senales(n_x, x, k_des, h_des)
    y_n = np.sum(producto)

    pds.graficar_discreta(n_x, x, titulo="x[k]", etiqueta_x="k", etiqueta_y="x[k]", ax=ejes[0])
    pds.graficar_discreta(k_des, h_des, titulo=f"h[{n}-k]", etiqueta_x="k", etiqueta_y=f"h[{n}-k]",
                          ax=ejes[1], color="C1")
    pds.graficar_discreta(k_prod, producto, titulo=f"producto: suma = y[{n}] = {y_n}", etiqueta_x="k",
                          etiqueta_y="x[k]·h[n-k]", ax=ejes[2], color="C3")
    for eje in ejes:
        eje.set_xlim(-3, 6)
    return y_n


figura, ejes = plt.subplots(3, 3, figsize=(14, 8))
for fila, n in enumerate([0, 2, 4]):
    mostrar_paso(n, ejes[fila])
plt.tight_layout()
plt.show()

Observa cómo la $h$ invertida "entra" por la izquierda, recorre la señal $x$ y sale por la derecha. Mientras no se
enciman, el producto es cero; por eso la salida sólo es distinta de cero en $L_x + L_h - 1 = 6$ muestras.

Comprobemos que juntando todos los pasos obtenemos lo mismo que la función de la biblioteca:

In [ ]:
y_pasos = []
for n in range(0, 6):
    k_inv, h_inv = pds.invertir_tiempo(n_h, h)
    k_des, h_des = pds.desplazar(k_inv, h_inv, n)
    k_prod, producto = pds.multiplicar_senales(n_x, x, k_des, h_des)
    y_pasos.append(np.sum(producto))

n_y, y = pds.convolucion(n_x, x, n_h, h)
print("paso a paso :", y_pasos)
print("biblioteca  :", y)
pds.comprobar_iguales(y_pasos, y, "invertir y deslizar da lo mismo que pds.convolucion")

## 3. Qué hace un kernel en 1D

En procesamiento de señales a $h$ también se le llama **kernel** o filtro. Su forma decide qué le pasa a la señal.
Probemos tres kernels sencillos sobre una señal escalonada con ruido:

| Kernel | Valores | Efecto |
|---|---|---|
| promedio | $\{1/5, 1/5, 1/5, 1/5, 1/5\}$ | suaviza: quita el ruido pero redondea los escalones |
| diferencia | $\{1, -1\}$ | aproxima la derivada: es casi cero donde la señal es constante y grande donde cambia |
| retardo | $\delta[n-10]$ | desplaza la señal 10 muestras sin cambiar su forma |

In [ ]:
generador = np.random.default_rng(10)
n = np.arange(0, 100)
escalones = np.zeros(100)
escalones[30:70] = 2.0
escalones[70:] = 1.0
x = escalones + 0.15 * generador.standard_normal(100)

h_promedio = np.ones(5) / 5
h_diferencia = np.array([1.0, -1.0])
h_retardo = np.zeros(11)
h_retardo[10] = 1.0

figura, ejes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
pds.graficar_discreta(n, x, titulo="señal con ruido", ax=ejes[0])
for eje, h_k, nombre in zip(ejes[1:], [h_promedio, h_diferencia, h_retardo],
                            ["promedio de 5", "diferencia [1, -1]", "retardo de 10 muestras"]):
    n_y, y = pds.convolucion(n, x, np.arange(len(h_k)), h_k)
    pds.graficar_discreta(n_y, y, titulo=f"convolución con {nombre}", ax=eje, color="C3")
plt.tight_layout()
plt.show()

Dos detalles que conviene notar:

- La salida es **más larga** que la entrada ($L_x + L_h - 1$). Las muestras extra del principio y del final son el
  transitorio: los momentos en que el kernel sólo se encima parcialmente con la señal.
- La diferencia marca los cambios, pero también amplifica el ruido. Por eso en la práctica primero se suaviza y luego se deriva.

## 4. Convoluciones largas: reverberación

Cuando aplaudes en una sala, el sonido rebota en paredes, piso y techo, y te llegan muchísimas copias retrasadas y
atenuadas. Esa sala es un sistema LTI, y su respuesta al impulso se parece a un ruido que se va apagando. Si
convolucionamos cualquier sonido con esa $h[n]$, suena como si se hubiera grabado en esa sala.

Aquí $h$ tiene miles de muestras, y nuestra función con ciclos sería muy lenta. Usamos `np.convolve`, que hace la
misma cuenta. Primero comprobamos, con señales cortas, que las dos dan el mismo resultado.

In [ ]:
x_corta = generador.standard_normal(50)
h_corta = generador.standard_normal(20)
_, y_nuestra = pds.convolucion(np.arange(50), x_corta, np.arange(20), h_corta)
pds.comprobar_iguales(y_nuestra, np.convolve(x_corta, h_corta), "np.convolve da los mismos valores")

In [ ]:
from IPython.display import display
fs = 8000

# Sonido seco: tres notas cortas con envolvente que se apaga rápido
partes = []
for F in [440, 554, 659]:
    n_s, t_s, tono = pds.senoidal_digital(F, fs, 0.15)
    partes.append(tono * np.exp(-t_s / 0.03))
    partes.append(np.zeros(pds.indice_de_muestra(0.25, fs)))
seco = np.concatenate(partes)

# Respuesta al impulso de una "sala": 0.6 s de ruido que decae exponencialmente
N_h = pds.indice_de_muestra(0.6, fs)              # número de muestras de h
t_h = np.arange(N_h) / fs                         # instante de cada muestra
h_sala = generador.standard_normal(N_h) * np.exp(-t_h / 0.15)
h_sala[0] = 1.0                                   # el sonido directo

con_sala = np.convolve(seco, h_sala)
print(f"seco: {len(seco)} muestras, h: {len(h_sala)} muestras, salida: {len(con_sala)} muestras")

figura, ejes = plt.subplots(3, 1, figsize=(10, 7))
pds.graficar_continua(np.arange(len(seco)) / fs, seco, titulo="sonido seco", ax=ejes[0])
pds.graficar_continua(t_h, h_sala, titulo="h[n] de la sala", ax=ejes[1])
pds.graficar_continua(np.arange(len(con_sala)) / fs, con_sala, titulo="sonido con la sala", ax=ejes[2])
plt.tight_layout()
plt.show()

print("Seco:")
display(pds.reproducir(seco, fs))
print("Con la sala:")
display(pds.reproducir(con_sala, fs))

## 5. Convolución en 2D

Con imágenes la idea es la misma, pero con dos índices: fila $m$ y columna $n$.

$$y[m, n] = \sum_{i}\sum_{j} h[i, j]\;x[m-i,\, n-j]$$

El kernel $h$ es una matriz pequeña (por ejemplo $3\times3$). Para obtener un pixel de salida se invierte el kernel
en las dos direcciones (se gira 180°), se coloca sobre la vecindad del pixel, se multiplica elemento a elemento y se suma.

### Un pixel a mano

Usemos una imagen de $5\times5$ y un kernel de $3\times3$ que no es simétrico, para que se note la inversión.

In [ ]:
imagen = np.array([[1, 2, 3, 4, 5],
                   [6, 7, 8, 9, 10],
                   [11, 12, 13, 14, 15],
                   [16, 17, 18, 19, 20],
                   [21, 22, 23, 24, 25]], dtype=float)
kernel = np.array([[1, 0, 0],
                   [0, 0, 0],
                   [0, 0, -1]], dtype=float)

# Pixel de salida en la fila 2, columna 2 (el centro), usando el modo "igual"
# Paso 1: invertir el kernel en las dos direcciones (girarlo 180 grados)
kernel_girado = kernel[::-1, ::-1]
print("kernel girado:\n", kernel_girado)

# Paso 2: tomar la vecindad de 3x3 alrededor del pixel (2, 2)
vecindad = imagen[1:4, 1:4]
print("vecindad:\n", vecindad)

# Paso 3: multiplicar elemento a elemento y sumar
producto = vecindad * kernel_girado
valor = np.sum(producto)
print("y[2, 2] =", valor)

Este kernel calcula "el pixel de arriba a la izquierda menos el de abajo a la derecha": $7 - 19 = -12$. Si hubiéramos
olvidado girar el kernel, obtendríamos $19 - 7 = +12$. Ese giro es justamente lo que distingue la convolución de la
correlación.

### Todos los pixeles con cuatro ciclos

Repitiendo lo anterior para cada pixel (y tomando como cero lo que queda fuera de la imagen) obtenemos la convolución
completa. Son cuatro ciclos anidados: dos para recorrer la imagen y dos para recorrer el kernel.

In [ ]:
filas, columnas = imagen.shape
P, Q = kernel.shape
salida = np.zeros((filas, columnas))

for m in range(filas):
    for n in range(columnas):
        suma = 0.0
        for i in range(P):
            for j in range(Q):
                # en modo "igual" el centro del kernel queda sobre el pixel (m, n)
                fila_x = m - (i - P // 2)
                columna_x = n - (j - Q // 2)
                if 0 <= fila_x < filas and 0 <= columna_x < columnas:
                    suma = suma + kernel[i, j] * imagen[fila_x, columna_x]
        salida[m, n] = suma

print(salida)
pds.comprobar_iguales(salida, pds.convolucion_2d(imagen, kernel), "los cuatro ciclos dan lo mismo que pds.convolucion_2d")

### Tamaño de la salida

Igual que en 1D, la convolución **completa** es más grande que la imagen: $(M+P-1)\times(N+Q-1)$. Con imágenes casi
siempre queremos una salida del mismo tamaño, así que nos quedamos con la parte central (modo `"igual"`).

In [ ]:
print("completa:", pds.convolucion_2d(imagen, kernel, modo="completa").shape)
print("igual   :", pds.convolucion_2d(imagen, kernel, modo="igual").shape)

### Cómo lo hace la biblioteca

Con una foto de $600\times512$ y un kernel de $9\times9$, los cuatro ciclos harían unos 25 millones de vueltas: varios
minutos. La función de la biblioteca usa la otra forma de ver la convolución, la **suma de copias**: por cada elemento
del kernel suma una copia de toda la imagen, desplazada y multiplicada. Sólo hace $P\times Q$ vueltas.

In [ ]:
import inspect
print(inspect.getsource(pds.convolucion_2d))

## 6. Kernels para imágenes

| Kernel | Suma de sus elementos | Efecto | Tipo de filtro |
|---|---|---|---|
| promedio | 1 | suaviza, desenfoca | pasa-bajas |
| gaussiano | 1 | suaviza de forma más natural | pasa-bajas |
| Sobel | 0 | detecta bordes en una dirección | pasa-altas |
| laplaciano | 0 | detecta bordes en todas las direcciones | pasa-altas |

Que la suma sea 1 significa que una zona de color uniforme no cambia. Que sea 0 significa que una zona uniforme
se vuelve 0, y sólo sobreviven los cambios.

In [ ]:
print("promedio 3x3:\n", pds.kernel_promedio(3))
print("gaussiano 5x5 (sigma = 1):\n", np.round(pds.kernel_gaussiano(5, 1), 3))
print("Sobel horizontal:\n", pds.kernel_sobel("horizontal"))
print("laplaciano:\n", pds.kernel_laplaciano())

In [ ]:
foto = pds.cargar_imagen("../datos/grace_hopper.jpg")

suave_promedio = pds.convolucion_2d(foto, pds.kernel_promedio(9))
suave_gauss = pds.convolucion_2d(foto, pds.kernel_gaussiano(9, 2))
gx = pds.convolucion_2d(foto, pds.kernel_sobel("horizontal"))
gy = pds.convolucion_2d(foto, pds.kernel_sobel("vertical"))
magnitud_bordes = np.sqrt(gx ** 2 + gy ** 2)        # bordes en cualquier dirección

figura, ejes = plt.subplots(2, 3, figsize=(15, 10))
pds.mostrar_imagen(foto, titulo="original", ax=ejes[0, 0])
pds.mostrar_imagen(suave_promedio, titulo="promedio 9x9", ax=ejes[0, 1])
pds.mostrar_imagen(suave_gauss, titulo="gaussiano 9x9, sigma = 2", ax=ejes[0, 2])
pds.mostrar_imagen(np.abs(gx), titulo="Sobel horizontal: bordes verticales", ax=ejes[1, 0])
pds.mostrar_imagen(np.abs(gy), titulo="Sobel vertical: bordes horizontales", ax=ejes[1, 1])
pds.mostrar_imagen(magnitud_bordes, titulo="magnitud del gradiente", ax=ejes[1, 2])
plt.tight_layout()
plt.show()

### Realce (hacer más nítida una imagen)

El laplaciano resalta los bordes. Si se lo **restamos** a la imagen original, los bordes se acentúan y la imagen se ve
más nítida:

$$y = x - c\,(x * \text{laplaciano})$$

con $c$ controlando la intensidad del efecto.

In [ ]:
bordes_lap = pds.convolucion_2d(foto, pds.kernel_laplaciano())
nitida = foto - 1.0 * bordes_lap

# acercamiento a la cara: filas y columnas de 150 a 349
figura, ejes = plt.subplots(1, 2, figsize=(11, 5.5))
pds.mostrar_imagen(foto[150:350, 150:350], titulo="original", ax=ejes[0])
pds.mostrar_imagen(np.clip(nitida[150:350, 150:350], 0, 1), titulo="realzada (c = 1)", ax=ejes[1])
plt.tight_layout()
plt.show()

## 7. Convolución y frecuencia

En el notebook 08 vimos que convolucionar en el tiempo equivale a multiplicar en frecuencia. En 2D pasa lo mismo:

$$y = x * h \quad\Longleftrightarrow\quad Y[k, l] = X[k, l]\,H[k, l]$$

siempre que las FFT se calculen con el tamaño de la convolución completa. Comprobémoslo:

In [ ]:
x_pequena = generador.random((20, 30))
h_pequeno = pds.kernel_sobel("horizontal")
tamano = (20 + 3 - 1, 30 + 3 - 1)

Y_espacio = np.fft.fft2(pds.convolucion_2d(x_pequena, h_pequeno, modo="completa"))
Y_frecuencia = np.fft.fft2(x_pequena, tamano) * np.fft.fft2(h_pequeno, tamano)
pds.comprobar_iguales(Y_espacio, Y_frecuencia, "FFT2(x * h) = FFT2(x) · FFT2(h)")

Eso nos permite ver qué hace cada kernel mirando su espectro. Rellenamos el kernel con ceros hasta $64\times64$ y
calculamos la magnitud de su FFT centrada:

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(14, 4.5))
kernels = [(pds.kernel_promedio(5), "promedio 5x5"),
           (pds.kernel_sobel("horizontal"), "Sobel horizontal"),
           (pds.kernel_laplaciano(), "laplaciano")]
for eje, (k, nombre) in zip(ejes, kernels):
    H = np.fft.fftshift(np.abs(np.fft.fft2(k, (64, 64))))
    pds.mostrar_imagen(H, titulo=f"|H| de {nombre}", ax=eje)
plt.tight_layout()
plt.show()

- El **promedio** deja pasar el centro (frecuencias bajas) y atenúa las orillas: es pasa-bajas.
- **Sobel** vale cero en la columna central y crece hacia los lados: deja pasar los cambios horizontales.
- El **laplaciano** vale cero en el centro y crece en todas las direcciones: es pasa-altas.

## Resumen

| | 1D | 2D |
|---|---|---|
| fórmula | $y[n] = \sum_k x[k]\,h[n-k]$ | $y[m,n] = \sum_i\sum_j h[i,j]\,x[m-i,n-j]$ |
| tamaño completo | $L_x + L_h - 1$ | $(M+P-1)\times(N+Q-1)$ |
| a mano | invertir $h$ y deslizar | girar el kernel 180° y deslizar |
| biblioteca | `pds.convolucion` (con índices) | `pds.convolucion_2d` (modo `"igual"` o `"completa"`) |
| señales largas | `np.convolve` | la biblioteca ya es rápida |
| en frecuencia | $Y = X\cdot H$ | $Y = X\cdot H$ con FFT2 |

## Comprueba tu comprensión

1. ¿Cuántas muestras tiene la convolución de una señal de 100 muestras con un kernel de 7?
2. Un kernel cuyos elementos suman 0, ¿qué le hace a una zona de la imagen con intensidad constante?
3. ¿Por qué convolucionar con $\{1, -1\}$ se parece a derivar?
4. Una imagen de $256\times256$ se convoluciona con un kernel de $5\times5$ en modo completo. ¿Qué tamaño tiene la salida?

<details><summary>Ver respuestas</summary>

1. $100 + 7 - 1 = 106$.
2. La vuelve cero.
3. Porque $y[n] = x[n] - x[n-1]$, que es la diferencia entre muestras consecutivas: la versión discreta de la pendiente.
4. $260\times260$.
</details>